# 12 — Query expansion: the vocabulary gap

> **Run order.** This notebook is step 12 of the pipeline. Earlier steps must
> have run at least once. See [`notebooks/README.md`](README.md).
>
> All reusable logic lives in `src/analyst/` — the package is unit-tested and
> type-checked, and these notebooks orchestrate it and show the results.

Four embedding models and a hybrid BM25 retriever all failed at roughly the same
place, which is the clue: when swapping the technique changes nothing, the
problem is not the technique.

**The questions and the filings do not use the same words.** That is measured
below, not asserted — and once it is visible, the fix is obvious and was already
sitting in the codebase.

In [1]:
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

from sqlalchemy import select

from analyst import evaluation as ev
from analyst.config import get_settings
from analyst.db import session_scope
from analyst.models import ElementRow

settings = get_settings()
questions = ev.load_questions(settings.data_dir / "benchmark" / "questions.jsonl")
print(f"{len(questions)} questions   benchmark {ev.bench_sha(questions)}")

44 questions   benchmark 2c4aedf3dcb75f7e

## The evidence: a question and its own answer share almost no words

`matched_as` is how the figure is printed in the report. If the question
contained it, BM25 would be a strong play. Count how many do.

In [2]:
have_figure = sum(1 for q in questions if q.matched_as and q.matched_as in q.question)
print(f"questions containing the figure they ask for: {have_figure} of {len(questions)}")

rows = []
with session_scope() as s:
    for q in questions:
        text = s.execute(
            select(ElementRow.text).where(ElementRow.element_id == q.expected_element_ids[0])
        ).scalar()
        if not text:
            continue
        qw = {w.strip(".,?'s").lower() for w in q.question.split() if len(w) > 3}
        ew = {w.strip(".,?'s").lower() for w in text.split() if len(w) > 3}
        rows.append({"ticker": q.ticker, "shared_words": len(qw & ew), "question_words": len(qw)})

overlap = pd.DataFrame(rows)
print(f"\nmedian shared words between a question and the element answering it: "
      f"{overlap['shared_words'].median():.0f}")
overlap.groupby("ticker")["shared_words"].describe()[["count", "mean", "min", "max"]].round(1)

questions containing the figure they ask for: 0 of 44


median shared words between a question and the element answering it: 2

,count,mean,min,max
ticker,,,,
HDFCBANK,2.0,1.5,1.0,2.0
ICICIBANK,10.0,1.4,0.0,3.0
RELIANCE,8.0,1.9,1.0,3.0
SUNPHARMA,24.0,1.8,1.0,3.0


### What that looks like concretely

The question asks for *total revenue in FY2024*. The filing prints *Revenue from
contracts with customers* under *Year ended March 31, 2024*. Nothing lexical
matches, and nothing dense matches either — an encoder trained on general
English has no reason to put those close together.

In [3]:
sun = next(q for q in questions if q.ticker == "SUNPHARMA" and q.concept == "Total Revenue")
with session_scope() as s:
    text = s.execute(
        select(ElementRow.text).where(ElementRow.element_id == sun.expected_element_ids[0])
    ).scalar()

print("QUESTION:", sun.question)
print("\nANSWER ELEMENT:")
print((text or "")[:260].replace("\n", " | "))

QUESTION:

What was Sun Pharmaceutical Industries's total revenue in FY2024?


ANSWER ELEMENT:

 | Year ended | March 31, 2024 | Year ended | March 31, 2023 | Revenue from contracts with customers (Refer note 53) | 477,584.5 | 432,788.7 | Other operating revenues* | 7,384.0 | 6,068.1 |  | 484,968.5 | 438,856.8

## The fix was already in the repo

`analyst.benchmark.CONCEPT_ALIASES` maps each concept to the labels Indian
annual reports actually print — it is how notebook 06 located every anchor in
the first place. The retriever simply never used it at query time.

`analyst.retrievers.expand_query` adds those labels plus the date form a filing
uses. Indian fiscal years end 31 March, so "FY2024" becomes "year ended
March 31, 2024".

In [4]:
from analyst.retrievers import expand_query

print("PLAIN   :", sun.question)
print("EXPANDED:", expand_query(sun))

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PLAIN   :

What was Sun Pharmaceutical Industries's total revenue in FY2024?

EXPANDED:

What was Sun Pharmaceutical Industries's total revenue in FY2024? total revenue revenue from operations total income revenue from contracts total revenue from operations year ended March 31, 2024

## Measure it

Four configurations on the identical index and the identical 44 questions.
Everything goes into the same ledger, so these rows sit beside the dense
baseline and the hybrid run.

In [5]:
from analyst.retrievers import dense, hybrid, open_hybrid, open_store

MODEL = "bge-small"
embedder, store = open_store(settings, MODEL)
_, sparse, hstore = open_hybrid(settings, MODEL)


def record(label: str, search, deep: bool = True) -> ev.Run:
    cfg = ev.RunConfig(retriever=label, model=MODEL, filters="ticker+year",
                       limit=max(ev.K_VALUES), points=store.count())
    run = ev.build_run(
        cfg,
        ev.evaluate(questions, search, limit=max(ev.K_VALUES)),
        questions,
        deep=ev.evaluate(questions, search, limit=max(ev.DEPTHS)) if deep else None,
    )
    ev.append_run(run)
    print(f"{label:<16} R@5 {run.metrics.recall_at[5]:.3f}  "
          f"ceiling@200 {run.depth_curve.get(200, 0):.3f}")
    return run


runs = {
    "dense+expand": record("dense+expand", dense(embedder, store, "ticker+year", expand=True)),
    "hybrid+expand": record("hybrid+expand",
                            hybrid(embedder, sparse, hstore, "ticker+year", expand=True)),
}

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


dense+expand     R@5 0.068  ceiling@200 0.682

hybrid+expand    R@5 0.068  ceiling@200 0.727

## The comparison

Read the **ceiling** column, not just R@5. Shallow recall on 44 questions moves
in steps of 0.023 — one question — so it is a noisy way to judge anything.
Recall at depth 200 is the honest measure of whether the right evidence is being
surfaced at all, and it is what caps any reranker added later.

In [6]:
ledger = ev.load_runs()
wanted = ("dense", "hybrid", "dense+expand", "hybrid+expand")
best = {}
for r in ledger:
    if r.config.retriever in wanted and r.config.model == MODEL \
            and r.config.filters == "ticker+year":
        best[r.config.retriever] = r

table = pd.DataFrame([best[k].row() for k in wanted if k in best])
print(table.to_string(index=False))

print()
curves = pd.DataFrame({k: best[k].depth_curve for k in wanted if k in best}).T
print(curves.rename_axis("retriever").to_string())

                             run     retriever     model     filters    R@1    R@3    R@5   R@10    MRR   pR@5  points  p50_ms    bench           git
        dense-bge-small-02c4b4ed         dense bge-small ticker+year 0.0227 0.0455 0.0455 0.0909 0.0392 0.0909    9982    84.8 2c4aedf3 3e65907-dirty
       hybrid-bge-small-ee1a298f        hybrid bge-small ticker+year 0.0227 0.0682 0.0682 0.1136 0.0449 0.1136    9982    90.0 2c4aedf3 3e65907-dirty
 dense+expand-bge-small-b60d0b39  dense+expand bge-small ticker+year 0.0455 0.0455 0.0682 0.1136 0.0559 0.1364    9982    87.7 2c4aedf3 2f3c9a3-dirty
hybrid+expand-bge-small-6510b044 hybrid+expand bge-small ticker+year 0.0455 0.0455 0.0682 0.0909 0.0544 0.1136    9982    91.1 2c4aedf3 2f3c9a3-dirty

                  1       5       10      20      50      100     200
retriever                                                            
dense          0.0227  0.0455  0.0909  0.1364  0.2273  0.2727  0.4318
hybrid         0.0227  0.0909  0.1136  0.1136  0.2045  0.2727  0.3636
dense+expand   0.0455  0.0682  0.1136  0.1591  0.3636  0.4773  0.6818
hybrid+expand  0.0000  0.0455  0.1136  0.1591  0.4773  0.6136  0.7273

## Where it still fails

Growth questions span two annual reports and score zero everywhere. They are a
different problem — multi-document reasoning, not retrieval — and no amount of
query rewriting fixes a question whose answer is not in any single chunk.

In [7]:
res = ev.evaluate(questions, dense(embedder, store, "ticker+year", expand=True),
                  limit=max(ev.DEPTHS))
df = pd.DataFrame([r.model_dump() for r in res])
print(df.groupby("question_type")["rank"].agg(n="size", found="count").to_string())
print()
print(df.groupby("ticker")["rank"].agg(n="size", found="count").to_string())

                n  found
question_type           
growth         10      5
value_lookup   34     25

            n  found
ticker              
HDFCBANK    2      1
ICICIBANK  10      7
RELIANCE    8      5
SUNPHARMA  24     17

## The ledger

In [8]:
print(ev.write_leaderboard(ev.load_runs()))

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\results\leaderboard.md